In [1]:
!pip install rechunker -q

In [19]:
client.shutdown()

KeyboardInterrupt: 

In [1]:
from dask.distributed import Client
client = Client()
client.cluster.scale(10)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,Workers: 4
Total threads: 4,Total memory: 14.54 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41365,Workers: 4
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,Total threads: 4
Started: Just now,Total memory: 14.54 GiB
Comm: tcp://127.0.0.1:46679,Total threads: 1
Dashboard: /user/jkingslake/load%20NCs/proxy/41169/status,Memory: 3.63 GiB
Nanny: tcp://127.0.0.1:41809,


2026-05-29 13:09:25,347 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:26,957 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:27,031 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:31,617 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:32,935 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:34,282 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:36,910 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:38,426 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:40,222 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:42,205 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:44,957 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:47,756 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:49,189 - distributed.nanny - WARNING - Restarting worker
2026-05-29 13:09:50,026 - distributed.

In [ ]:
client.cluster.scale(40)

Process Dask Worker process (from Nanny):
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/srv/conda/envs/notebook/lib/python3.12/asyncio/base_events.py", line 691, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/distributed/nanny.py", line 985, in run
    await worker.finished()
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/distributed/core.py", line 494, in finished
    await self._event_finished.wait()
  File "/srv/conda/envs/notebook/lib/python3.12/asyncio/locks.py", line 212, in wait
    await fut
asyncio.exceptions.CancelledError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.12/multiprocessing/process.

In [11]:
import xarray as xr
import s3fs
from rechunker import rechunk
import zarr

fs = s3fs.S3FileSystem()

SOURCE_PATH = "s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/t2m_all_06.zarr"
TARGET_PATH = "s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/t2m_all_time_series_07.zarr"
TEMP_PATH   = "s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/t2m_all_temp_07.zarr"
MAX_MEM     = "2GB"

source_store = s3fs.S3Map(SOURCE_PATH, s3=fs)
target_store = s3fs.S3Map(TARGET_PATH, s3=fs)
temp_store   = s3fs.S3Map(TEMP_PATH,   s3=fs)


# Open as zarr group directly, not xarray
source_group = zarr.open_consolidated(source_store, mode='r')

source_array = source_group['t2mcorr']

# # Build target chunks as before
# target_chunks = {
#     var: {"time": source_group[var].shape[0], "x": 75, "y": 75}
#     for var in source_group
# }

# print("Target chunks:", target_chunks)

ds = xr.open_zarr(SOURCE_PATH, consolidated=True)

plan = rechunk(
    source_array,      
    target_chunks=(ds.time.size, 75, 75),
    max_mem=MAX_MEM,
    target_store=target_store,
    temp_store=temp_store,
)
plan.execute()

print(f"Done! Written to {TARGET_PATH}")

# Clean up temp
fs.rm(TEMP_PATH, recursive=True)
print("Temp store deleted.")

# Verify
ds_out = xr.open_zarr(target_store, consolidated=True)
print(ds_out)

2026-05-29 13:22:27,906 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('copy_read_to_intermediate-476230ef85dd2e723dab54edfd5a5eeb', 12716))" coro=<Worker.execute() done, defined at /srv/conda/envs/notebook/lib/python3.12/site-packages/distributed/worker_state_machine.py:3606>> ended with CancelledError
2026-05-29 13:22:28,069 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('copy_read_to_intermediate-476230ef85dd2e723dab54edfd5a5eeb', 50273))" coro=<Worker.execute() done, defined at /srv/conda/envs/notebook/lib/python3.12/site-packages/distributed/worker_state_machine.py:3606>> ended with CancelledError


KeyboardInterrupt: 

In [5]:
# Clear target and temp stores before running
if fs.exists(TARGET_PATH):
    fs.rm(TARGET_PATH, recursive=True)
    print("Cleared target store")

if fs.exists(TEMP_PATH):
    fs.rm(TEMP_PATH, recursive=True)
    print("Cleared temp store")

Cleared target store
Cleared temp store


In [12]:
ds.time.size

17167

In [13]:
ds.chunk(x=75, y=75)

KeyboardInterrupt: 